In [1]:
# Importaciones necesarias
using DifferentialEquations
using SciMLSensitivity
using Optimization
using OptimizationOptimisers
using Lux
using ComponentArrays
using Random
using Zygote


using Plots, Printf

using ComponentArrays

using DataFrames, Parquet2, Dates


ENV["GKSwstype"] = "100"

In [2]:
file_path = "data/clean_dataset.parquet"

data = DataFrame(Parquet2.readfile(file_path))

Row,State,ID_Period,Start_Date,Report_Date,Days_In_Period,New_Deaths,New_Cases,Accumulated_Cases,Accumulated_Deaths,Active,Recovered,People_Hospitalized,Accumulated_Deaths_Rate,New_Deaths_Rate
,String,Int64,DateTime,DateTime,Int64,Int64,Int64,Int64,Int64,Float64?,Float64?,Float64?,Float64,Float64
1,Alabama,1,2020-04-12T23:18:15,2020-04-12T23:18:15,1,93,3667,3667,93,missing,missing,437.0,0.0253613,0.0253613
2,Alabama,2,2020-04-13T23:07:54,2020-04-13T23:07:54,1,6,203,3870,99,missing,missing,457.0,0.0255814,0.0295567
3,Alabama,3,2020-04-14T23:33:31,2020-04-14T23:33:31,1,15,171,4041,114,missing,missing,493.0,0.0282108,0.0877193
4,Alabama,4,2020-04-15T22:56:51,2020-04-15T22:56:51,1,4,266,4307,118,missing,missing,525.0,0.0273973,0.0150376
5,Alabama,5,2020-04-16T23:30:51,2020-04-16T23:30:51,1,15,158,4465,133,missing,missing,553.0,0.0297872,0.0949367
6,Alabama,6,2020-04-17T23:30:52,2020-04-17T23:30:52,1,15,92,4557,148,missing,missing,594.0,0.0324775,0.163043
7,Alabama,7,2020-04-18T22:32:47,2020-04-18T22:32:47,1,5,231,4788,153,missing,missing,620.0,0.0319549,0.021645
8,Alabama,8,2020-04-19T23:41:01,2020-04-19T23:41:01,1,4,190,4978,157,missing,missing,641.0,0.0315388,0.0210526
9,Alabama,9,2020-04-20T23:36:47,2020-04-20T23:36:47,1,6,185,5163,163,missing,missing,641.0,0.0315708,0.0324324


In [3]:
n = 10
top_n_states = groupby(data, :State)
top_n_states = combine(top_n_states, 
    :ID_Period => maximum => :Max_ID_Period) 

sort!(top_n_states, :Max_ID_Period, rev=true)
top_n_states = first(top_n_states, n)
df_top_n = filter(row -> row.State in top_n_states.State, data)
df_grouped = groupby(df_top_n, :State)

train_dfs = DataFrame.(collect(df_grouped))
show(train_dfs[1], allrows=true, allcols=true)

978×14 DataFrame
 Row │ State     ID_Period  Start_Date           Report_Date          Days_In_Period  New_Deaths  New_Cases  Accumulated_Cases  Accumulated_Deaths  Active     Recovered  People_Hospitalized  Accumulated_Deaths_Rate  New_Deaths_Rate 
     │ String    Int64      DateTime             DateTime             Int64           Int64       Int64      Int64              Int64               Float64?   Float64?   Float64?             Float64                  Float64         
─────┼──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │ Arkansas          1  2020-04-12T23:18:15  2020-04-12T23:18:15               1          27       1280               1280                  27      886.0      367.0                130.0                0.0210938      0.0210938
   2 │ Arkansas          2  2020-04-13T23:07:54  2020-

In [ ]:
f_activation(v) = 1 ./ (1 .+ exp.(.-v)) #Biological activation function

const nn = Chain(
    Dense(6 => 16, f_activation),
    Dense(16 => 16, f_activation),
    Dense(16 => 4, f_activation)
)

rng_nn        = MersenneTwister(666)
nn_ps_, nn_st = Lux.setup(rng_nn, nn)
nn_ps      = Float64.(ComponentArray(nn_ps_)) #Simpliphy str

# Nota: Ya no tiene el "!" al final y solo recibe (u, p, t)
function UDE_SIER(u, p, t)
    S, E, I, R, C, D = u

    # Pasamos 'p' (que es θ) a la red
    params, _ = nn(u, p, nn_st)
    
    β = abs(params[1]) + 1e-5
    σ = abs(params[2]) + 1e-5
    γ = abs(params[3]) + 1e-5
    μ = abs(params[4]) + 1e-5
    
    N = 1.0 

    dS = -β * S * I / N            
    dE =  β * S * I / N - σ * E    
    dI =  σ * E - (γ + μ) * I      
    dR =  γ * I                    
    dC =  σ * E                    
    dD =  μ * I                    
    
    # Zygote es 100% compatible con la creación de arreglos nuevos
    return [dS, dE, dI, dR, dC, dD]
end

param_history = Vector{Vector{Float64}}()
loss_history  = Vector{Float64}()

function loss(θ, opts_dfs)

    dfs = opts_dfs.dfs

    model_st = opts_dfs.st

    total_loss = zero(eltype(θ)) # Inicialización compatible con Zygote

    for df in dfs
        t = df.ID_Period
        tspan = (first(t), last(t))
            
        I0 = df.Accumulated_Cases[1] 
        E0 = df.New_Cases[1]
        R0 = 0.0
        S0 = 1.0 - (I0 + E0 + R0)
        C0 = df.Accumulated_Cases[1]
        D0 = df.Accumulated_Deaths[1]
    
        T = eltype(θ) 
        
        # Aplicamos T() a TODO para evitar choques de tipos (Type Instability)
        u0 = T.([S0, E0, I0, R0, C0, D0])

        dff_ude = ODEProblem(UDE_SIER, u0, tspan, θ)
        # UDE solver (The use of Rodas5P garantee stiffness seal)
        sol = solve(dff_ude, Rodas5P(), saveat=t,
                    ensealg=InterpolatingAdjoint(autojacvec=ZygoteVJP()))
    
        # If the integration fail, penalize by maintaining "p" dependency to calculate the gradient
        if sol.retcode != ReturnCode.Success
            return 1e6 + 1e4 * sum(abs2, θ)
        end
        
        û = Array(sol)
        
        # Prevent NaN
        if any(isnan, û)
            return 1e6 + 1e4 * sum(abs2, θ)
        end

        C_pred = û[5, :] 
        D_pred = û[6, :] 
        
        C_nuevos_pred = C_pred[2:end] .- C_pred[1:end-1]
        C_nuevos_real = df.New_Cases[2:end]
        
        # Cálculo de Errores Cuadráticos Medios (MSE)
        loss_C = sum(abs2, C_pred .- df.Accumulated_Cases)
        loss_D = sum(abs2, D_pred .- df.Accumulated_Deaths)
        loss_Nuevos = sum(abs2, C_nuevos_pred .- C_nuevos_real)
        
        total_loss += (loss_C + 5.0 * loss_D + loss_Nuevos)
    end
    
    return total_loss
end

callback_entrenamiento = function (state, loss_val)
    println("Iteración finalizada. Pérdida: ", round(loss_val, sigdigits=6))
    return false 
end



# ==============================================================================
# 4. BUCLE DE ENTRENAMIENTO
# ==============================================================================

function tuple_data(dfs_reales)
    datos_limpios = []
    for df in dfs_reales
        push!(datos_limpios, (
            ID_Period = Float64.(df.ID_Period),
            Accumulated_Cases = Float64.(df.Accumulated_Cases), # Ajusta el nombre de tu columna real aquí
            New_Cases = Float64.(df.New_Cases),          # Ajusta el nombre de tu columna real aquí
            Accumulated_Deaths = Float64.(df.Accumulated_Deaths) # Ajusta el nombre de tu columna real aquí
        ))
    end
    return datos_limpios
end

function train(df_list)
    println("Configurando el optimizador...")
    train_tuple = tuple_data(df_list) # Preprocesar los DataFrames para el optimizador
    p_optimizador = (dfs = train_tuple, st = nn_st)
    
    # AutoZygote hará el backward pass a través de nuestro closure
    opt_func = OptimizationFunction(loss, Optimization.AutoZygote())
    opt_prob = OptimizationProblem(opt_func, nn_ps, p_optimizador)
    
    println("Iniciando el entrenamiento (ADAM, 50 iteraciones)...")
    res = solve(opt_prob, OptimizationOptimisers.Adam(0.01), callback = callback_entrenamiento, maxiters = 500)
    
    println("\n¡Entrenamiento completado!")
    return res.u 
end

# Ejecución
nn_ŵ = train(train_dfs)


In [48]:
function extraer_parametros_dinamicos(dfs_reales, θ_opt, st_lux)
    resultados = []

    for (i, df) in enumerate(dfs_reales)
        t = df.ID_Period
        tspan = (first(t), last(t))
        
        I0 = df.Accumulated_Cases[1] 
        E0 = df.New_Cases[1]
        R0 = 0.0
        S0 = 1.0 - (I0 + E0 + R0)
        C0 = df.Accumulated_Cases[1]
        D0 = df.Accumulated_Deaths[1]
    
        u0 = Float64.([S0, E0, I0, R0, C0, D0])

        prob = ODEProblem(nn_dynamics!, u0, tspan, θ_opt)
        sol = solve(prob, Rodas5P(), saveat=t)
        
        # NUEVO: Te avisa si la red neuronal causó una inestabilidad matemática
        if sol.retcode != ReturnCode.Success
            println("⚠️ Advertencia en DF $i: La integración abortó prematuramente. Retcode: ", sol.retcode)
        end

        β_vals = Float64[]
        σ_vals = Float64[]
        γ_vals = Float64[]
        μ_vals = Float64[]

        for u_t in sol.u
            nn_out, _ = nn(u_t, θ_opt, st_lux)
            
            push!(β_vals, abs(nn_out[1]) + 1e-5)
            push!(σ_vals, abs(nn_out[2]) + 1e-5)
            push!(γ_vals, abs(nn_out[3]) + 1e-5)
            push!(μ_vals, abs(nn_out[4]) + 1e-5)
        end
        
        # CORRECCIÓN: Usar 'sol.t' en lugar de 't'. 
        # SciML garantiza que sol.t y sol.u tengan siempre la misma longitud.
        df_params = DataFrame(
            Periodo = sol.t, 
            Beta    = β_vals,
            Sigma   = σ_vals,
            Gamma   = γ_vals,
            Mu      = μ_vals
        )
        
        push!(resultados, (DF_Origen = "DataFrame $i", Parametros = df_params))
    end
    
    return resultados
end
# 2. Utilizar los pesos óptimos para reconstruir las tasas de las ecuaciones diferenciales
parametros_dinamicos = extraer_parametros_dinamicos(train_dfs, nn_ŵ, nn_st)

⚠️ Advertencia en DF 1: La integración abortó prematuramente. Retcode: Unstable
⚠️ Advertencia en DF 2: La integración abortó prematuramente. Retcode: Unstable
⚠️ Advertencia en DF 3: La integración abortó prematuramente. Retcode: Unstable
⚠️ Advertencia en DF 4: La integración abortó prematuramente. Retcode: Unstable
⚠️ Advertencia en DF 5: La integración abortó prematuramente. Retcode: Unstable
⚠️ Advertencia en DF 6: La integración abortó prematuramente. Retcode: Unstable
⚠️ Advertencia en DF 8: La integración abortó prematuramente. Retcode: Unstable
⚠️ Advertencia en DF 9: La integración abortó prematuramente. Retcode: Unstable
⚠️ Advertencia en DF 10: La integración abortó prematuramente. Retcode: Unstable


10-element Vector{Any}:
 (DF_Origen = "DataFrame 1", Parametros = 1×5 DataFrame
 Row │ Periodo  Beta      Sigma     Gamma     Mu       
     │ Float64  Float64   Float64   Float64   Float64  
─────┼─────────────────────────────────────────────────
   1 │     1.0  0.494426  0.503543  0.497038  0.495508)
 (DF_Origen = "DataFrame 2", Parametros = 1×5 DataFrame
 Row │ Periodo  Beta      Sigma     Gamma     Mu       
     │ Float64  Float64   Float64   Float64   Float64  
─────┼─────────────────────────────────────────────────
   1 │     1.0  0.494426  0.503543  0.497038  0.495508)
 (DF_Origen = "DataFrame 3", Parametros = 1×5 DataFrame
 Row │ Periodo  Beta      Sigma     Gamma     Mu       
     │ Float64  Float64   Float64   Float64   Float64  
─────┼─────────────────────────────────────────────────
   1 │     1.0  0.494426  0.503543  0.497038  0.495508)
 (DF_Origen = "DataFrame 4", Parametros = 1×5 DataFrame
 Row │ Periodo  Beta      Sigma     Gamma     Mu       
     │ Float64  Float64 

In [ ]:
# Red Neuronal Optimizada para evitar gradientes muertos (Vanishing Gradients)
const nn = Chain(
    Dense(6 => 16, tanh),
    Dense(16 => 16, tanh),
    Dense(16 => 4) # <-- ¡Crucial! Sin función de activación al final.
)

rng_nn        = MersenneTwister(666)
nn_ps_, nn_st = Lux.setup(rng_nn, nn)
nn_ps         = Float64.(ComponentArray(nn_ps_))

function UDE_SIER(u, p, t)
    S, E, I, R, C, D = u

    # CORRECCIÓN 1: Usamos 'p' (los pesos que recibe la función), no 'θ'
    params, _ = nn(u, p, nn_st)
    
    # Limitar a valores positivos sin usar exp() para evitar SegFaults
    β = abs(params[1]) + 1e-5
    σ = abs(params[2]) + 1e-5
    γ = abs(params[3]) + 1e-5
    μ = abs(params[4]) + 1e-5
    
    N = 1.0 

    # CORRECCIÓN 2: Creamos variables locales normales en lugar de usar du[...]
    dS = -β * S * I / N            
    dE =  β * S * I / N - σ * E    
    dI =  σ * E - (γ + μ) * I      
    dR =  γ * I                    
    dC =  σ * E                    
    dD =  μ * I                    
    
    # CORRECCIÓN 3: Retornamos explícitamente el nuevo estado como un vector
    return [dS, dE, dI, dR, dC, dD]
end

param_history = Vector{Vector{Float64}}()
loss_history  = Vector{Float64}()

callback_entrenamiento = function (state, loss_val)
    println("Pérdida: ", round(loss_val, sigdigits=6), " | Norma Pesos: ", round(sum(abs2, state.u), sigdigits=5))
    return false 
end
# ==============================================================================
# 2. FUNCIÓN LOSS PARA ZYGOTE
# ==============================================================================
function loss(θ, opts_dfs)
    dfs = opts_dfs.dfs
    st_local = opts_dfs.st 
    
    function UDE_SIER_local(u, p, t)
        S, E, I, R, C, D = u
        
        params, _ = nn(u, p, st_local) 
        
        β = abs(params[1]) + 1e-5
        σ = abs(params[2]) + 1e-5
        γ = abs(params[3]) + 1e-5
        μ = abs(params[4]) + 1e-5
        
        N = 1.0 

        dS = -β * S * I / N            
        dE =  β * S * I / N - σ * E    
        dI =  σ * E - (γ + μ) * I      
        dR =  γ * I                    
        dC =  σ * E                    
        dD =  μ * I                    
        
        return [dS, dE, dI, dR, dC, dD]
    end

    losses = map(dfs) do df
        t = df.ID_Period
        tspan = (first(t), last(t))
            
        I0 = df.Accumulated_Cases[1] 
        E0 = df.New_Cases[1]
        R0 = 0.0
        S0 = 1.0 - (I0 + E0 + R0)
        C0 = df.Accumulated_Cases[1]
        D0 = df.Accumulated_Deaths[1]
    
        u0 = eltype(θ).([S0, E0, I0, R0, C0, D0])

        prob = ODEProblem(UDE_SIER_local, u0, tspan, θ)
        
        # Eliminamos isoutofdomain para no interrumpir el cálculo del gradiente
        sol = solve(prob, Tsit5(), saveat=t,
                    sensealg = InterpolatingAdjoint(autojacvec=ZygoteVJP()))
    
        if sol.retcode != ReturnCode.Success
            return eltype(θ)(1e6) + eltype(θ)(1e4) * sum(abs2, θ)
        end
        
        # =========================================================
        # ¡LA CLAVE ESTÁ AQUÍ! Extraemos directamente de 'sol' 
        # sin usar Array() para no romper la memoria de Zygote
        # =========================================================
        C_pred = sol[5, :] 
        D_pred = sol[6, :] 
        
        C_nuevos_pred = C_pred[2:end] .- C_pred[1:end-1]
        C_nuevos_real = df.New_Cases[2:end]
        
        loss_C = sum(abs2, C_pred .- df.Accumulated_Cases)
        loss_D = sum(abs2, D_pred .- df.Accumulated_Deaths)
        loss_Nuevos = sum(abs2, C_nuevos_pred .- C_nuevos_real)
        
        return loss_C + 5.0 * loss_D + loss_Nuevos
    end
    
    return sum(losses)
end

function tuple_data(dfs_reales, pob_total)
    datos_limpios = []
    for df in dfs_reales
        push!(datos_limpios, (
            ID_Period = Float64.(df.ID_Period),
            # Dividimos por pob_total para que todos los valores estén entre 0.0 y 1.0
            Accumulated_Cases  = Float64.(df.Accumulated_Cases)  ./ pob_total,
            New_Cases          = Float64.(df.New_Cases)          ./ pob_total, 
            Accumulated_Deaths = Float64.(df.Accumulated_Deaths) ./ pob_total
        ))
    end
    return datos_limpios
end

function train(df_list, pob_total)
    println("Configurando el optimizador...")
    # Pasamos la población total para normalizar antes de entrenar
    train_tuple = tuple_data(df_list, pob_total) 
    p_optimizador = (dfs = train_tuple, st = nn_st)
    
    opt_func = OptimizationFunction(loss, Optimization.AutoZygote())
    opt_prob = OptimizationProblem(opt_func, nn_ps, p_optimizador)
    
    println("Iniciando el entrenamiento (ADAM, 200 iteraciones)...")
    
    # NUEVO: Learning rate reducido (0.005) y 200 iteraciones máximas
    res = solve(opt_prob, OptimizationOptimisers.Adam(), 
                callback = callback_entrenamiento,
                maxiters = 50)
    
    println("\n¡Entrenamiento completado!")
    return res.u 
end

N_TOTAL = 1_000_000.0 # Ajusta este valor según la población total de tus datos

# Ejecución
pesos_optimos = train(train_dfs, N_TOTAL)

Configurando el optimizador...
Iniciando el entrenamiento (ADAM, 200 iteraciones)...


┌ Warning: Using arrays or dicts to store parameters of different types can hurt performance.
│ Consider using tuples instead.
└ @ SciMLBase C:\Users\santi\.julia\packages\SciMLBase\O1HPI\src\performance_warnings.jl:32


Pérdida: 92679.2 | Norma Pesos: 90.931
Pérdida: 92443.5 | Norma Pesos: 90.936
Pérdida: 92193.2 | Norma Pesos: 90.955
Pérdida: 91876.4 | Norma Pesos: 90.971
Pérdida: 91479.5 | Norma Pesos: 90.986
Pérdida: 90980.4 | Norma Pesos: 91.004
Pérdida: 90334.8 | Norma Pesos: 91.024
Pérdida: 89468.4 | Norma Pesos: 91.047
Pérdida: 88243.0 | Norma Pesos: 91.076
Pérdida: 86472.5 | Norma Pesos: 91.108
Pérdida: 84318.7 | Norma Pesos: 91.142
Pérdida: 82655.2 | Norma Pesos: 91.174
Pérdida: 81307.7 | Norma Pesos: 91.201
Pérdida: 79331.7 | Norma Pesos: 91.228
Pérdida: 76713.5 | Norma Pesos: 91.25
Pérdida: 74667.6 | Norma Pesos: 91.271
Pérdida: 73776.6 | Norma Pesos: 91.292
Pérdida: 73261.8 | Norma Pesos: 91.312
Pérdida: 72978.8 | Norma Pesos: 91.332
Pérdida: 73327.4 | Norma Pesos: 91.351
Pérdida: 73359.1 | Norma Pesos: 91.369
Pérdida: 73263.0 | Norma Pesos: 91.388
Pérdida: 73137.0 | Norma Pesos: 91.406
Pérdida: 72880.0 | Norma Pesos: 91.426
Pérdida: 72631.3 | Norma Pesos: 91.445
Pérdida: 72373.0 | Norma P

In [ ]:
######################################################################
# MODELO SEIR-UDE (Neural ODE) PARA SERIES EPIDEMIOLÓGICAS MULTI-ESTADO
# Versión con ploteo en tiempo real y mejor debugging
######################################################################

# ====================================================================
# 1. DEPENDENCIAS
# ====================================================================
using DifferentialEquations
using SciMLSensitivity
using Optimization
using OptimizationOptimisers
using OptimizationOptimJL
using Lux
using ComponentArrays
using Random
using Zygote
using LinearAlgebra
using StaticArrays
using Plots
using Printf
using DataFrames
using Parquet2
using Dates
using JLD2

# Configurar backend de gráficas
gr()  # Usar backend GR en lugar de pyplot
ENV["GKSwstype"] = "100"

# ====================================================================
# 2. CONFIGURACIÓN / HIPERPARÁMETROS
# ====================================================================

Base.@kwdef struct ModelConfig
    eps_tasa::Float64     = 1e-5
    n_norm::Float64       = 1.0
    penalty_base::Float64 = 1e6
    penalty_reg::Float64  = 1e4
    w_deaths::Float64     = 5.0
end

Base.@kwdef struct TrainConfig
    seed::Int           = 666
    lr::Float64         = 0.005
    maxiters_adam::Int  = 200
    maxiters_lbfgs::Int = 100
    verbose_every::Int  = 10
    n_states::Int       = 1
    pob_total::Float64  = 1_000_000.0
    data_path::String   = "data/clean_dataset.parquet"
end

# ====================================================================
# 3. CARGA Y PREPROCESAMIENTO DE DATOS
# ====================================================================

function load_top_states(file_path::String, n::Int)
    println("  Cargando archivo: $file_path")
    flush(stdout)
    
    if !isfile(file_path)
        error("❌ Archivo no encontrado: $file_path")
    end
    
    data = DataFrame(Parquet2.readfile(file_path))
    println("  ✓ Datos cargados: $(size(data, 1)) filas")
    flush(stdout)

    summary = combine(groupby(data, :State), :ID_Period => maximum => :Max_ID_Period)
    sort!(summary, :Max_ID_Period, rev = true)
    top_states = first(summary, n).State

    df_top  = filter(row -> row.State in top_states, data)
    grouped = groupby(df_top, :State)
    dfs = DataFrame.(collect(grouped))
    
    println("  ✓ $(length(dfs)) estados seleccionados")
    for (i, df) in enumerate(dfs)
        println("    $i. $(df.State[1]) ($(size(df, 1)) períodos)")
    end
    flush(stdout)
    
    return dfs
end

function normalize_data(dfs, pob_total::Real)
    println("\nNormalizando datos (población total = $pob_total)...")
    flush(stdout)
    
    normalized = map(dfs) do df
        (
            State              = df.State[1],
            ID_Period          = Float64.(df.ID_Period),
            Accumulated_Cases  = Float64.(df.Accumulated_Cases)  ./ pob_total,
            New_Cases          = Float64.(df.New_Cases)          ./ pob_total,
            Accumulated_Deaths = Float64.(df.Accumulated_Deaths) ./ pob_total,
        )
    end
    
    println("  ✓ Datos normalizados")
    flush(stdout)
    return normalized
end

function initial_conditions(ds, ::Type{T}) where {T}
    I0 = T(ds.Accumulated_Cases[1])
    E0 = T(ds.New_Cases[1])
    R0 = T(0.0)
    C0 = T(ds.Accumulated_Cases[1])
    D0 = T(ds.Accumulated_Deaths[1])
    S0 = max(T(1.0) - (I0 + E0 + R0), T(0.0))
    return SVector{6,T}(S0, E0, I0, R0, C0, D0)
end

# ====================================================================
# 4. RED NEURONAL
# ====================================================================

@inline f_activation(v) = one(v) / (one(v) + exp(-v))

function build_nn(rng::AbstractRNG)
    nn = Chain(
        Dense(6 => 16, f_activation),
        Dense(16 => 16, f_activation),
        Dense(16 => 4),
    )
    ps_raw, st = Lux.setup(rng, nn)
    ps = ComponentArray{Float64}(ps_raw)
    println("  ✓ Red neuronal creada: $(sum(length, ps)) parámetros")
    flush(stdout)
    return nn, ps, st
end

# ====================================================================
# 5. DINÁMICA DEL SISTEMA (IN-PLACE)
# ====================================================================

@inline function sier_rates(nn, u, p, st, cfg::ModelConfig)
    raw, st_new = nn(u, p, st)
    β = abs(raw[1]) + cfg.eps_tasa
    σ = abs(raw[2]) + cfg.eps_tasa
    γ = abs(raw[3]) + cfg.eps_tasa
    μ = abs(raw[4]) + cfg.eps_tasa
    return β, σ, γ, μ, st_new
end

function make_dynamics(nn, st, cfg::ModelConfig)
    return function (du, u, p, t)
        S, E, I, R, C, D = u
        β, σ, γ, μ, _ = sier_rates(nn, u, p, st, cfg)

        du[1] = -β * S * I / cfg.n_norm
        du[2] =  β * S * I / cfg.n_norm - σ * E
        du[3] =  σ * E - (γ + μ) * I
        du[4] =  γ * I
        du[5] =  σ * E
        du[6] =  μ * I

        return nothing
    end
end

# ====================================================================
# 6. FUNCIÓN DE PÉRDIDA
# ====================================================================

function dataset_loss(θ, ds, dynamics, cfg::ModelConfig)
    t     = ds.ID_Period
    tspan = (first(t), last(t))
    u0    = Vector{eltype(θ)}(initial_conditions(ds, eltype(θ)))

    prob = ODEProblem(dynamics, u0, tspan, θ)
    sol  = solve(prob, Tsit5(), saveat = t,
                  sensealg = InterpolatingAdjoint(autojacvec = ZygoteVJP()),
                  abstol = 1e-8, reltol = 1e-6,
                  verbose = false)

    if sol.retcode != ReturnCode.Success
        return cfg.penalty_base + cfg.penalty_reg * sum(abs2, θ)
    end

    C_pred = sol[5, :]
    D_pred = sol[6, :]

    n = length(C_pred)
    ΔC_pred = [C_pred[i+1] - C_pred[i] for i in 1:n-1]
    ΔC_real = [ds.New_Cases[i+1] - ds.New_Cases[i] for i in 1:n-1]

    loss_C   = sum(abs2, C_pred .- ds.Accumulated_Cases)
    loss_D   = sum(abs2, D_pred .- ds.Accumulated_Deaths)
    loss_new = sum(abs2, ΔC_pred .- ΔC_real)

    return loss_C + cfg.w_deaths * loss_D + loss_new
end

function total_loss(θ, p)
    dynamics = make_dynamics(p.nn, p.st, p.cfg)
    return sum(ds -> dataset_loss(θ, ds, dynamics, p.cfg), p.dfs)
end

# ====================================================================
# 7. ENTRENAMIENTO
# ====================================================================

function train(df_list, mcfg::ModelConfig, tcfg::TrainConfig)
    println("\nConfigurando entrenamiento...")
    flush(stdout)
    
    rng = MersenneTwister(tcfg.seed)
    nn, θ0, st = build_nn(rng)

    train_data = normalize_data(df_list, tcfg.pob_total)
    p = (dfs = train_data, nn = nn, st = st, cfg = mcfg)

    loss_history = Float64[]
    iter_count = 0

    callback = function (state, loss_val)
        iter_count += 1
        push!(loss_history, loss_val)
        
        if iter_count == 1 || iter_count % tcfg.verbose_every == 0
            @printf("  Iter %4d | Pérdida = %.6e | ‖θ‖ = %.5e\n",
                    iter_count, loss_val, norm(state.u))
            flush(stdout)
        end
        return false
    end

    opt_func = OptimizationFunction(total_loss, Optimization.AutoZygote())

    println("\n--- Etapa 1: Adam ($(tcfg.maxiters_adam) iteraciones) ---")
    flush(stdout)
    
    prob1 = OptimizationProblem(opt_func, θ0, p)
    res1  = solve(prob1, OptimizationOptimisers.Adam(tcfg.lr),
                   callback = callback, maxiters = tcfg.maxiters_adam,
                   verbose = false)

    println("\n--- Etapa 2: LBFGS (refinamiento) ---")
    flush(stdout)
    
    prob2 = OptimizationProblem(opt_func, res1.u, p)
    res2  = solve(prob2, OptimizationOptimJL.LBFGS(),
                   callback = callback, maxiters = tcfg.maxiters_lbfgs,
                   verbose = false)

    println("\n✅ ¡Entrenamiento completado!")
    flush(stdout)
    
    return (params = res2.u, loss_history = loss_history,
            nn = nn, st = st, cfg = mcfg, train_data = train_data)
end

# ====================================================================
# 8. VISUALIZACIÓN Y DIAGNÓSTICO (CORREGIDO)
# ====================================================================

function plot_loss(history; outfile = "loss_history.png")
    println("\n📊 Generando gráfica de pérdida...")
    flush(stdout)
    
    try
        plt = plot(history, 
                   xlabel = "Iteración", 
                   ylabel = "Pérdida",
                   yscale = :log10, 
                   lw = 2, 
                   legend = false,
                   title = "Evolución de la pérdida durante el entrenamiento",
                   size = (800, 600),
                   color = :blue)
        
        savefig(plt, outfile)
        display(plt)  # Mostrar la gráfica
        println("  ✓ Gráfica guardada: $outfile")
        flush(stdout)
        return plt
    catch e
        @error "Error al generar gráfica de pérdida: $e"
        rethrow()
    end
end

function plot_fit(θ, ds, nn, st, cfg::ModelConfig; outdir = ".")
    println("  Procesando: $(ds.State)...")
    flush(stdout)
    
    try
        dynamics = make_dynamics(nn, st, cfg)
        t  = ds.ID_Period
        u0 = Vector{eltype(θ)}(initial_conditions(ds, eltype(θ)))

        prob = ODEProblem(dynamics, u0, (first(t), last(t)), θ)
        sol  = solve(prob, Tsit5(), saveat = t,
                     abstol = 1e-8, reltol = 1e-6)

        if sol.retcode != ReturnCode.Success
            @warn "  ⚠️  La solución ODE falló para $(ds.State)"
            return nothing
        end

        # Gráfica de casos
        p1 = plot(t, ds.Accumulated_Cases, 
                  label = "Reales", 
                  lw = 2,
                  marker = :circle, 
                  markersize = 3,
                  markeralpha = 0.5,
                  color = :blue)
        plot!(p1, t, sol[5, :], 
              label = "Predichos", 
              lw = 2, 
              ls = :dash,
              color = :red)
        xlabel!(p1, "Tiempo")
        ylabel!(p1, "Casos acumulados")
        title!(p1, "Casos Acumulados")

        # Gráfica de muertes
        p2 = plot(t, ds.Accumulated_Deaths, 
                  label = "Reales", 
                  lw = 2,
                  marker = :circle, 
                  markersize = 3,
                  markeralpha = 0.5,
                  color = :blue)
        plot!(p2, t, sol[6, :], 
              label = "Predichos", 
              lw = 2, 
              ls = :dash,
              color = :red)
        xlabel!(p2, "Tiempo")
        ylabel!(p2, "Muertes acumuladas")
        title!(p2, "Muertes Acumuladas")

        # Combinar gráficas
        plt = plot(p1, p2, 
                   layout = (2, 1), 
                   size = (800, 700),
                   title = "Ajuste del modelo - $(ds.State)",
                   legend = :topleft)

        # Guardar
        safe_name = replace(ds.State, " " => "_", "/" => "_", "\\" => "_")
        outfile = joinpath(outdir, "fit_$(safe_name).png")
        savefig(plt, outfile)
        
        # Mostrar
        display(plt)
        
        println("    ✓ Guardado: $outfile")
        flush(stdout)
        return plt
        
    catch e
        @error "Error al graficar $(ds.State): $e"
        return nothing
    end
end

function save_results(resultado; outfile = "trained_model.jld2")
    println("\n💾 Guardando resultados...")
    flush(stdout)
    
    try
        @save outfile params=Array(resultado.params) loss_history=resultado.loss_history
        println("  ✓ Modelo guardado en: $outfile")
        flush(stdout)
        return outfile
    catch e
        @error "Error al guardar resultados: $e"
        rethrow()
    end
end

# ====================================================================
# 9. EJECUCIÓN PRINCIPAL
# ====================================================================

function main()
    println("="^70)
    println("MODELO SEIR-UDE - Neural ODE para Series Epidemiológicas")
    println("="^70)
    flush(stdout)
    
    mcfg = ModelConfig()
    tcfg = TrainConfig()

    # Verificar archivo
    if !isfile(tcfg.data_path)
        error("❌ El archivo de datos no existe: $(tcfg.data_path)")
    end

    # Cargar datos
    println("\n📁 Cargando datos...")
    flush(stdout)
    df_list = load_top_states(tcfg.data_path, tcfg.n_states)
    
    # Entrenar
    println("\n🏋️  Entrenando modelo...")
    flush(stdout)
    resultado = train(df_list, mcfg, tcfg)

    # Guardar
    save_results(resultado)

    # Graficar
    println("\n📈 Generando visualizaciones...")
    flush(stdout)
    
    plot_loss(resultado.loss_history, outfile = "loss_history.png")
    
    for (i, ds) in enumerate(resultado.train_data)
        println("\nGráfica $i/$(length(resultado.train_data)):")
        plot_fit(resultado.params, ds, resultado.nn, resultado.st, mcfg)
    end

    println("\n" * "="^70)
    println("✅ PROCESO COMPLETADO EXITOSAMENTE")
    println("="^70)
    println("\nArchivos generados:")
    println("  - trained_model.jld2 (parámetros)")
    println("  - loss_history.png (pérdida)")
    println("  - fit_*.png (ajustes por estado)")
    flush(stdout)
    
    return resultado
end

# Ejecutar
println("\n🚀 Iniciando ejecución...\n")
flush(stdout)

global resultado = main()
global pesos_optimos = resultado.params

println("\n📊 Resumen final:")
println("  - Parámetros entrenados: $(length(pesos_optimos))")
println("  - Pérdida final: $(resultado.loss_history[end])")
println("  - Estados procesados: $(length(resultado.train_data))")
flush(stdout)


🚀 Iniciando ejecución...

MODELO SEIR-UDE - Neural ODE para Series Epidemiológicas

📁 Cargando datos...
  Cargando archivo: data/clean_dataset.parquet
  ✓ Datos cargados: 40702 filas
  ✓ 10 estados seleccionados
    1. Arkansas (978 períodos)
    2. California (995 períodos)
    3. Colorado (936 períodos)
    4. Maryland (894 períodos)
    5. New Jersey (959 períodos)
    6. New York (987 períodos)
    7. North Dakota (992 períodos)
    8. Puerto Rico (969 períodos)
    9. Tennessee (997 períodos)
    10. Texas (988 períodos)

🏋️  Entrenando modelo...

Configurando entrenamiento...
  ✓ Red neuronal creada: 452 parámetros

Normalizando datos (población total = 1.0e6)...
  ✓ Datos normalizados

--- Etapa 1: Adam (200 iteraciones) ---
  Iter    1 | Pérdida = 9.374286e+04 | ‖θ‖ = 6.01693e+00


In [ ]:
######################################################################
# MODELO SEIR-UDE (Neural ODE) PARA SERIES EPIDEMIOLÓGICAS MULTI-ESTADO
# Versión con ploteo en tiempo real y mejor debugging
######################################################################

# ====================================================================
# 1. DEPENDENCIAS
# ====================================================================
using DifferentialEquations
using SciMLSensitivity
using Optimization
using OptimizationOptimisers
using OptimizationOptimJL
using Lux
using ComponentArrays
using Random
using Zygote
using LinearAlgebra
using StaticArrays
using Plots
using Printf
using DataFrames
using Parquet2
using Dates
using JLD2

# Configurar backend de gráficas
gr()  # Usar backend GR en lugar de pyplot
ENV["GKSwstype"] = "100"

# ====================================================================
# 2. CONFIGURACIÓN / HIPERPARÁMETROS
# ====================================================================

Base.@kwdef struct ModelConfig
    eps_tasa::Float64     = 1e-5
    n_norm::Float64       = 1.0
    penalty_base::Float64 = 1e6
    penalty_reg::Float64  = 1e4
    w_deaths::Float64     = 5.0
end

Base.@kwdef struct TrainConfig
    seed::Int           = 666
    lr::Float64         = 0.005
    maxiters_adam::Int  = 200
    maxiters_lbfgs::Int = 100
    verbose_every::Int  = 10
    n_states::Int       = 1
    pob_total::Float64  = 1_000_000.0
    data_path::String   = "data/clean_dataset.parquet"
end

# ====================================================================
# 3. CARGA Y PREPROCESAMIENTO DE DATOS
# ====================================================================

function load_top_states(file_path::String, n::Int)
    println("  Cargando archivo: $file_path")
    flush(stdout)
    
    if !isfile(file_path)
        error("❌ Archivo no encontrado: $file_path")
    end
    
    data = DataFrame(Parquet2.readfile(file_path))
    println("  ✓ Datos cargados: $(size(data, 1)) filas")
    flush(stdout)

    summary = combine(groupby(data, :State), :ID_Period => maximum => :Max_ID_Period)
    sort!(summary, :Max_ID_Period, rev = true)
    top_states = first(summary, n).State

    df_top  = filter(row -> row.State in top_states, data)
    grouped = groupby(df_top, :State)
    dfs = DataFrame.(collect(grouped))
    
    println("  ✓ $(length(dfs)) estados seleccionados")
    for (i, df) in enumerate(dfs)
        println("    $i. $(df.State[1]) ($(size(df, 1)) períodos)")
    end
    flush(stdout)
    
    return dfs
end

function normalize_data(dfs, pob_total::Real)
    println("\nNormalizando datos (población total = $pob_total)...")
    flush(stdout)
    
    normalized = map(dfs) do df
        (
            State              = df.State[1],
            ID_Period          = Float64.(df.ID_Period),
            Accumulated_Cases  = Float64.(df.Accumulated_Cases)  ./ pob_total,
            New_Cases          = Float64.(df.New_Cases)          ./ pob_total,
            Accumulated_Deaths = Float64.(df.Accumulated_Deaths) ./ pob_total,
        )
    end
    
    println("  ✓ Datos normalizados")
    flush(stdout)
    return normalized
end

function initial_conditions(ds, ::Type{T}) where {T}
    I0 = T(ds.Accumulated_Cases[1])
    E0 = T(ds.New_Cases[1])
    R0 = T(0.0)
    C0 = T(ds.Accumulated_Cases[1])
    D0 = T(ds.Accumulated_Deaths[1])
    S0 = max(T(1.0) - (I0 + E0 + R0), T(0.0))
    return SVector{6,T}(S0, E0, I0, R0, C0, D0)
end

# ====================================================================
# 4. RED NEURONAL
# ====================================================================

@inline f_activation(v) = one(v) / (one(v) + exp(-v))

function build_nn(rng::AbstractRNG)
    nn = Chain(
        Dense(6 => 16, f_activation),
        Dense(16 => 16, f_activation),
        Dense(16 => 4),
    )
    ps_raw, st = Lux.setup(rng, nn)
    ps = ComponentArray{Float64}(ps_raw)
    println("  ✓ Red neuronal creada: $(sum(length, ps)) parámetros")
    flush(stdout)
    return nn, ps, st
end

# ====================================================================
# 5. DINÁMICA DEL SISTEMA (IN-PLACE)
# ====================================================================

@inline function sier_rates(nn, u, p, st, cfg::ModelConfig)
    raw, st_new = nn(u, p, st)
    β = abs(raw[1]) + cfg.eps_tasa
    σ = abs(raw[2]) + cfg.eps_tasa
    γ = abs(raw[3]) + cfg.eps_tasa
    μ = abs(raw[4]) + cfg.eps_tasa
    return β, σ, γ, μ, st_new
end

function make_dynamics(nn, st, cfg::ModelConfig)
    return function (du, u, p, t)
        S, E, I, R, C, D = u
        β, σ, γ, μ, _ = sier_rates(nn, u, p, st, cfg)

        du[1] = -β * S * I / cfg.n_norm
        du[2] =  β * S * I / cfg.n_norm - σ * E
        du[3] =  σ * E - (γ + μ) * I
        du[4] =  γ * I
        du[5] =  σ * E
        du[6] =  μ * I

        return nothing
    end
end

# ====================================================================
# 6. FUNCIÓN DE PÉRDIDA
# ====================================================================

function dataset_loss(θ, ds, dynamics, cfg::ModelConfig)
    t     = ds.ID_Period
    tspan = (first(t), last(t))
    u0    = Vector{eltype(θ)}(initial_conditions(ds, eltype(θ)))

    prob = ODEProblem(dynamics, u0, tspan, θ)
    sol  = solve(prob, Tsit5(), saveat = t,
                  sensealg = InterpolatingAdjoint(autojacvec = ZygoteVJP()),
                  abstol = 1e-8, reltol = 1e-6,
                  verbose = false)

    if sol.retcode != ReturnCode.Success
        return cfg.penalty_base + cfg.penalty_reg * sum(abs2, θ)
    end

    C_pred = sol[5, :]
    D_pred = sol[6, :]

    n = length(C_pred)
    ΔC_pred = [C_pred[i+1] - C_pred[i] for i in 1:n-1]
    ΔC_real = [ds.New_Cases[i+1] - ds.New_Cases[i] for i in 1:n-1]

    loss_C   = sum(abs2, C_pred .- ds.Accumulated_Cases)
    loss_D   = sum(abs2, D_pred .- ds.Accumulated_Deaths)
    loss_new = sum(abs2, ΔC_pred .- ΔC_real)

    return loss_C + cfg.w_deaths * loss_D + loss_new
end

function total_loss(θ, p)
    dynamics = make_dynamics(p.nn, p.st, p.cfg)
    return sum(ds -> dataset_loss(θ, ds, dynamics, p.cfg), p.dfs)
end

# ====================================================================
# 7. ENTRENAMIENTO
# ====================================================================

function train(df_list, mcfg::ModelConfig, tcfg::TrainConfig)
    println("\nConfigurando entrenamiento...")
    flush(stdout)
    
    rng = MersenneTwister(tcfg.seed)
    nn, θ0, st = build_nn(rng)

    train_data = normalize_data(df_list, tcfg.pob_total)
    p = (dfs = train_data, nn = nn, st = st, cfg = mcfg)

    loss_history = Float64[]
    iter_count = 0

    callback = function (state, loss_val)
        iter_count += 1
        push!(loss_history, loss_val)
        
        if iter_count == 1 || iter_count % tcfg.verbose_every == 0
            @printf("  Iter %4d | Pérdida = %.6e | ‖θ‖ = %.5e\n",
                    iter_count, loss_val, norm(state.u))
            flush(stdout)
        end
        return false
    end

    opt_func = OptimizationFunction(total_loss, Optimization.AutoZygote())

    println("\n--- Etapa 1: Adam ($(tcfg.maxiters_adam) iteraciones) ---")
    flush(stdout)
    
    prob1 = OptimizationProblem(opt_func, θ0, p)
    res1  = solve(prob1, OptimizationOptimisers.Adam(tcfg.lr),
                   callback = callback, maxiters = tcfg.maxiters_adam,
                   verbose = false)

    println("\n--- Etapa 2: LBFGS (refinamiento) ---")
    flush(stdout)
    
    prob2 = OptimizationProblem(opt_func, res1.u, p)
    res2  = solve(prob2, OptimizationOptimJL.LBFGS(),
                   callback = callback, maxiters = tcfg.maxiters_lbfgs,
                   verbose = false)

    println("\n✅ ¡Entrenamiento completado!")
    flush(stdout)
    
    return (params = res2.u, loss_history = loss_history,
            nn = nn, st = st, cfg = mcfg, train_data = train_data)
end

# ====================================================================
# 8. VISUALIZACIÓN Y DIAGNÓSTICO (CORREGIDO)
# ====================================================================

function plot_loss(history; outfile = "loss_history.png")
    println("\n📊 Generando gráfica de pérdida...")
    flush(stdout)
    
    try
        plt = plot(history, 
                   xlabel = "Iteración", 
                   ylabel = "Pérdida",
                   yscale = :log10, 
                   lw = 2, 
                   legend = false,
                   title = "Evolución de la pérdida durante el entrenamiento",
                   size = (800, 600),
                   color = :blue)
        
        savefig(plt, outfile)
        display(plt)  # Mostrar la gráfica
        println("  ✓ Gráfica guardada: $outfile")
        flush(stdout)
        return plt
    catch e
        @error "Error al generar gráfica de pérdida: $e"
        rethrow()
    end
end

function plot_fit(θ, ds, nn, st, cfg::ModelConfig; outdir = ".")
    println("  Procesando: $(ds.State)...")
    flush(stdout)
    
    try
        dynamics = make_dynamics(nn, st, cfg)
        t  = ds.ID_Period
        u0 = Vector{eltype(θ)}(initial_conditions(ds, eltype(θ)))

        prob = ODEProblem(dynamics, u0, (first(t), last(t)), θ)
        sol  = solve(prob, Tsit5(), saveat = t,
                     abstol = 1e-8, reltol = 1e-6)

        if sol.retcode != ReturnCode.Success
            @warn "  ⚠️  La solución ODE falló para $(ds.State)"
            return nothing
        end

        # Gráfica de casos
        p1 = plot(t, ds.Accumulated_Cases, 
                  label = "Reales", 
                  lw = 2,
                  marker = :circle, 
                  markersize = 3,
                  markeralpha = 0.5,
                  color = :blue)
        plot!(p1, t, sol[5, :], 
              label = "Predichos", 
              lw = 2, 
              ls = :dash,
              color = :red)
        xlabel!(p1, "Tiempo")
        ylabel!(p1, "Casos acumulados")
        title!(p1, "Casos Acumulados")

        # Gráfica de muertes
        p2 = plot(t, ds.Accumulated_Deaths, 
                  label = "Reales", 
                  lw = 2,
                  marker = :circle, 
                  markersize = 3,
                  markeralpha = 0.5,
                  color = :blue)
        plot!(p2, t, sol[6, :], 
              label = "Predichos", 
              lw = 2, 
              ls = :dash,
              color = :red)
        xlabel!(p2, "Tiempo")
        ylabel!(p2, "Muertes acumuladas")
        title!(p2, "Muertes Acumuladas")

        # Combinar gráficas
        plt = plot(p1, p2, 
                   layout = (2, 1), 
                   size = (800, 700),
                   title = "Ajuste del modelo - $(ds.State)",
                   legend = :topleft)

        # Guardar
        safe_name = replace(ds.State, " " => "_", "/" => "_", "\\" => "_")
        outfile = joinpath(outdir, "fit_$(safe_name).png")
        savefig(plt, outfile)
        
        # Mostrar
        display(plt)
        
        println("    ✓ Guardado: $outfile")
        flush(stdout)
        return plt
        
    catch e
        @error "Error al graficar $(ds.State): $e"
        return nothing
    end
end

function save_results(resultado; outfile = "trained_model.jld2")
    println("\n💾 Guardando resultados...")
    flush(stdout)
    
    try
        @save outfile params=Array(resultado.params) loss_history=resultado.loss_history
        println("  ✓ Modelo guardado en: $outfile")
        flush(stdout)
        return outfile
    catch e
        @error "Error al guardar resultados: $e"
        rethrow()
    end
end

# ====================================================================
# 9. EJECUCIÓN PRINCIPAL
# ====================================================================

function main()
    println("="^70)
    println("MODELO SEIR-UDE - Neural ODE para Series Epidemiológicas")
    println("="^70)
    flush(stdout)
    
    mcfg = ModelConfig()
    tcfg = TrainConfig()

    # Verificar archivo
    if !isfile(tcfg.data_path)
        error("❌ El archivo de datos no existe: $(tcfg.data_path)")
    end

    # Cargar datos
    println("\n📁 Cargando datos...")
    flush(stdout)
    df_list = load_top_states(tcfg.data_path, tcfg.n_states)
    
    # Entrenar
    println("\n🏋️  Entrenando modelo...")
    flush(stdout)
    resultado = train(df_list, mcfg, tcfg)

    # Guardar
    save_results(resultado)

    # Graficar
    println("\n📈 Generando visualizaciones...")
    flush(stdout)
    
    plot_loss(resultado.loss_history, outfile = "loss_history.png")
    
    for (i, ds) in enumerate(resultado.train_data)
        println("\nGráfica $i/$(length(resultado.train_data)):")
        plot_fit(resultado.params, ds, resultado.nn, resultado.st, mcfg)
    end

    println("\n" * "="^70)
    println("✅ PROCESO COMPLETADO EXITOSAMENTE")
    println("="^70)
    println("\nArchivos generados:")
    println("  - trained_model.jld2 (parámetros)")
    println("  - loss_history.png (pérdida)")
    println("  - fit_*.png (ajustes por estado)")
    flush(stdout)
    
    return resultado
end

# Ejecutar
println("\n🚀 Iniciando ejecución...\n")
flush(stdout)

global resultado = main()
global pesos_optimos = resultado.params

println("\n📊 Resumen final:")
println("  - Parámetros entrenados: $(length(pesos_optimos))")
println("  - Pérdida final: $(resultado.loss_history[end])")
println("  - Estados procesados: $(length(resultado.train_data))")
flush(stdout)


🚀 Iniciando ejecución...

MODELO SEIR-UDE - Neural ODE para Series Epidemiológicas

📁 Cargando datos...
  Cargando archivo: data/clean_dataset.parquet
  ✓ Datos cargados: 40702 filas
  ✓ 1 estados seleccionados
    1. Tennessee (997 períodos)

🏋️  Entrenando modelo...

Configurando entrenamiento...
  ✓ Red neuronal creada: 452 parámetros

Normalizando datos (población total = 1.0e6)...
  ✓ Datos normalizados

--- Etapa 1: Adam (200 iteraciones) ---
  Iter    1 | Pérdida = 2.276851e+03 | ‖θ‖ = 6.01693e+00
  Iter   10 | Pérdida = 7.566942e+02 | ‖θ‖ = 6.01748e+00
  Iter   20 | Pérdida = 7.968088e+02 | ‖θ‖ = 6.00642e+00
  Iter   30 | Pérdida = 7.896990e+02 | ‖θ‖ = 6.00507e+00
  Iter   40 | Pérdida = 7.802761e+02 | ‖θ‖ = 6.00605e+00
  Iter   50 | Pérdida = 7.789252e+02 | ‖θ‖ = 6.00629e+00
  Iter   60 | Pérdida = 7.785855e+02 | ‖θ‖ = 6.00615e+00
  Iter   70 | Pérdida = 7.785971e+02 | ‖θ‖ = 6.00571e+00
  Iter   80 | Pérdida = 7.785014e+02 | ‖θ‖ = 6.00519e+00
  Iter   90 | Pérdida = 7.784063e